In [ ]:
""" RFM Segmentation + Churn Prediction
UCI Online Retail Dataset (synthetic)
Portfolio project for Business Analyst roles """


In [5]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.facecolor':   '#f8f9fa',
    'figure.facecolor': 'white',
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# Colour palette
PRIMARY   = '#1a1a2e'
ACCENT2   = '#0f3460'
HIGHLIGHT = '#e94560'
GOLD      = '#f5a623'
TEAL      = '#00b4d8'
MINT      = '#48cae4'
GREY      = '#6c757d'
GREEN     = '#2d6a4f'
ORANGE    = '#f4845f'

SEG_COLORS = {
    'Champions':         '#2d6a4f',
    'Loyal Customers':   '#40916c',
    'At Risk':           '#e94560',
    'Hibernating':       '#f4845f',
    'New Customers':     '#0f3460',
    'Promising':         '#00b4d8',
    'Potential Loyalist':'#48cae4',
    'Lost':              '#adb5bd',
}

In [ ]:
df = pd.read_excel('Online Retail.xlsx', engine='openpyxl')
print(f"File loaded")
df.head()


In [ ]:
import pandas as pd

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df = df.dropna(subset=['CustomerID'])
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df['CustomerID'] = df['CustomerID'].astype(str)

print(f"Clean records : {len(df):,}")
print(f"Unique customers : {df['CustomerID'].nunique():,}")
print(f"Date range : {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}")
df.head()

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 2. RFM CALCULATION
# ══════════════════════════════════════════════════════════════════
snapshot = df['InvoiceDate'].max() + pd.Timedelta(days=1)
 
rfm = (df.groupby('CustomerID')
         .agg(
             Recency   = ('InvoiceDate', lambda x: (snapshot - x.max()).days),
             Frequency = ('InvoiceDate', 'count'),
             Monetary  = ('Revenue',     'sum')
         )
         .reset_index())
 
# Score 1–5 (5 = best)
rfm['R_Score'] = pd.qcut(rfm['Recency'],   5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print("RFM table (first 10 rows):")
rfm.head(10)

In [ ]:
def assign_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 4 and f >= 4 and m >= 4:    return 'Champions'
    if r >= 3 and f >= 3:               return 'Loyal Customers'
    if r >= 4 and f <= 2:               return 'New Customers'
    if r >= 3 and f >= 2 and m >= 2:    return 'Potential Loyalist'
    if r >= 3 and f == 1:               return 'Promising'
    if r == 2 and f >= 2:               return 'At Risk'
    if r <= 2 and f <= 2 and m >= 3:    return 'Hibernating'
    return 'Lost'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

print("Segment counts:")
print(rfm['Segment'].value_counts().to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 2. CHURN PREDICTION MODEL
# ══════════════════════════════════════════════════════════════════
# Churn definition: no purchase in last 90 days

rfm['Churned'] = (rfm['Recency'] > 90).astype(int)
print(f"Churn rate: {rfm['Churned'].mean():.1%}  ({rfm['Churned'].sum()} churned / {len(rfm)} total)")

features = ['Recency','Frequency','Monetary','R_Score','F_Score','M_Score','RFM_Score']
X = rfm[features]
y = rfm['Churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler     = StandardScaler()
X_train_s  = scaler.fit_transform(X_train)
X_test_s   = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=200, max_depth=6,
                                random_state=42, class_weight='balanced')
model.fit(X_train_s, y_train)

y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:,1]
auc     = roc_auc_score(y_test, y_proba)

print(f"\nModel AUC: {auc:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Active','Churned']))

rfm['Churn_Prob'] = model.predict_proba(scaler.transform(X))[:,1]

In [ ]:
# Visualizations 
# RFM Segment Overview 

seg_order  = ['Champions','Loyal Customers','Potential Loyalist',
              'Promising','New Customers','At Risk','Hibernating','Lost']
seg_counts = rfm['Segment'].value_counts().reindex(seg_order).dropna()
seg_colors = [SEG_COLORS[s] for s in seg_counts.index]

fig1, axes = plt.subplots(2, 3, figsize=(18, 11))
fig1.suptitle('RFM Customer Segmentation Analysis', fontsize=18, fontweight='bold', y=0.98)
plt.subplots_adjust(hspace=0.45, wspace=0.35)

# 1a – Horizontal bar
ax = axes[0,0]
ax.sharex  # reset
ax2_ = fig1.add_subplot(2,3,1)
fig1.delaxes(axes[0,0])
bars = ax2_.barh(seg_counts.index[::-1], seg_counts.values[::-1],
                 color=[SEG_COLORS[s] for s in seg_counts.index[::-1]],
                 edgecolor='white', height=0.65)
for bar, val in zip(bars, seg_counts.values[::-1]):
    ax2_.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
              f'{val:,}  ({val/len(rfm):.0%})', va='center', fontsize=8.5)
ax2_.set_xlabel('Customers'); ax2_.set_xlim(0, seg_counts.max()*1.3)
ax2_.set_title('Segment Distribution', fontweight='bold')
ax2_.set_facecolor('#f8f9fa'); ax2_.grid(alpha=0.3)

# 1b – Donut
ax = axes[0,1]
wedges, _, ats = ax.pie(seg_counts.values, colors=seg_colors,
                         autopct=lambda p: f'{p:.0f}%' if p>4 else '',
                         startangle=140, pctdistance=0.72,
                         wedgeprops=dict(width=0.55, edgecolor='white', linewidth=1.5))
for at in ats: at.set_fontsize(8); at.set_color('white'); at.set_fontweight('bold')
ax.set_title('Segment Share', fontweight='bold')
patches = [mpatches.Patch(color=SEG_COLORS[s], label=s) for s in seg_counts.index]
ax.legend(handles=patches, loc='center', bbox_to_anchor=(0.5,-0.12), ncol=2, fontsize=7, frameon=False)

# 1c – RFM score histogram
ax = axes[0,2]
ax.hist(rfm['RFM_Score'], bins=12, color=TEAL, edgecolor='white', alpha=0.85)
ax.axvline(rfm['RFM_Score'].mean(), color=HIGHLIGHT, lw=2, ls='--',
           label=f"Mean: {rfm['RFM_Score'].mean():.1f}")
ax.set_xlabel('RFM Score (3–15)'); ax.set_ylabel('Customers')
ax.set_title('RFM Score Distribution', fontweight='bold'); ax.legend()

# 1d – Avg monetary by segment
ax = axes[1,0]
seg_mon = rfm.groupby('Segment')['Monetary'].mean().reindex(seg_order).dropna()
ax.bar(range(len(seg_mon)), seg_mon.values,
       color=[SEG_COLORS[s] for s in seg_mon.index], edgecolor='white')
ax.set_xticks(range(len(seg_mon)))
ax.set_xticklabels(seg_mon.index, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Avg Revenue (£)'); ax.set_title('Avg Revenue by Segment', fontweight='bold')

# 1e – Recency vs Frequency scatter
ax = axes[1,1]
for seg in seg_order:
    sub = rfm[rfm['Segment']==seg]
    if len(sub): ax.scatter(sub['Recency'], sub['Frequency'],
                             c=SEG_COLORS[seg], label=seg, alpha=0.5, s=18)
ax.set_xlabel('Recency (days)'); ax.set_ylabel('Frequency')
ax.set_title('Recency vs Frequency', fontweight='bold')

# 1f – M score boxplot by segment
ax = axes[1,2]
seg_data   = [rfm[rfm['Segment']==s]['Monetary'].values for s in seg_order if s in rfm['Segment'].values]
seg_labels = [s for s in seg_order if s in rfm['Segment'].values]
bp = ax.boxplot(seg_data, patch_artist=True, vert=True,
                medianprops=dict(color='white', lw=2))
for patch, s in zip(bp['boxes'], seg_labels):
    patch.set_facecolor(SEG_COLORS[s]); patch.set_alpha(0.8)
ax.set_xticklabels(seg_labels, rotation=40, ha='right', fontsize=7.5)
ax.set_ylabel('Revenue (£)'); ax.set_title('Revenue Distribution', fontweight='bold')

fig1.savefig('rfm_overview.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# Churn Prediction Model Results 

fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
fig2.suptitle(f'Churn Prediction Model  |  AUC = {auc:.3f}  |  Churn Rate = {rfm["Churned"].mean():.1%}',
              fontsize=16, fontweight='bold')
plt.subplots_adjust(hspace=0.50, wspace=0.38)

# 2a – ROC curve
ax = axes2[0,0]
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax.plot(fpr, tpr, color=HIGHLIGHT, lw=2.5, label=f'AUC = {auc:.3f}')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax.fill_between(fpr, tpr, alpha=0.08, color=HIGHLIGHT)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold'); ax.legend()

# 2b – Confusion matrix
ax = axes2[0,1]
cm = confusion_matrix(y_test, y_pred)
cmap = LinearSegmentedColormap.from_list('cm',['#f8f9fa', ACCENT2])
sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
            xticklabels=['Active','Churned'], yticklabels=['Active','Churned'],
            linewidths=2, linecolor='white', cbar=False, annot_kws={'size':14,'weight':'bold'})
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix', fontweight='bold')

# 2c – Feature importance
ax = axes2[0,2]
imp = pd.Series(model.feature_importances_, index=features).sort_values()
colors_imp = [HIGHLIGHT if v >= imp.quantile(0.6) else TEAL for v in imp.values]
ax.barh(imp.index, imp.values, color=colors_imp, edgecolor='white')
ax.set_xlabel('Importance'); ax.set_title('Feature Importance', fontweight='bold')

# 2d – Churn prob by segment
ax = axes2[1,0]
sc = rfm.groupby('Segment')['Churn_Prob'].mean().reindex(seg_order).dropna().sort_values()
colors_sc = ['#2d6a4f' if v<0.3 else ORANGE if v<0.6 else HIGHLIGHT for v in sc.values]
ax.barh(sc.index, sc.values, color=colors_sc, edgecolor='white', height=0.6)
ax.axvline(0.5, color=HIGHLIGHT, lw=1.5, ls='--', alpha=0.7)
ax.set_xlabel('Avg Churn Probability'); ax.set_xlim(0,1)
ax.set_title('Churn Risk by Segment', fontweight='bold')
for i, v in enumerate(sc.values): ax.text(v+0.01, i, f'{v:.0%}', va='center', fontsize=9)

# 2e – Score distribution
ax = axes2[1,1]
ax.hist(rfm[rfm['Churned']==0]['Churn_Prob'], bins=25,
        alpha=0.7, color=GREEN,    label='Active',  density=True, edgecolor='white')
ax.hist(rfm[rfm['Churned']==1]['Churn_Prob'], bins=25,
        alpha=0.7, color=HIGHLIGHT, label='Churned', density=True, edgecolor='white')
ax.set_xlabel('Predicted Churn Probability'); ax.set_ylabel('Density')
ax.set_title('Churn Score Distribution', fontweight='bold'); ax.legend()

# 2f – Top 10 at-risk customers
ax = axes2[1,2]
ax.axis('off')
top10 = (rfm[rfm['Churned']==0]
           .nlargest(10,'Churn_Prob')
           [['CustomerID','Recency','Frequency','Monetary','Churn_Prob']]
           .copy())
top10['Monetary']   = top10['Monetary'].map('£{:.0f}'.format)
top10['Churn_Prob'] = top10['Churn_Prob'].map('{:.0%}'.format)
top10.columns = ['Customer','Recency','Freq','Revenue','Churn%']
tbl = ax.table(cellText=top10.values, colLabels=top10.columns,
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1,1.4)
for (r,c), cell in tbl.get_celld().items():
    if r==0: cell.set_facecolor(PRIMARY); cell.set_text_props(color='white',weight='bold')
    elif r%2==0: cell.set_facecolor('#f0f4f8')
    cell.set_edgecolor('white')
ax.set_title('Top 10 At-Risk Active Customers', fontweight='bold', pad=12)

fig1.savefig('rfm_churn_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Marketing and Retention Insights 

fig3, axes3 = plt.subplots(2, 3, figsize=(18, 10))
fig3.suptitle('Marketing & Retention Insights', fontsize=18, fontweight='bold')
plt.subplots_adjust(hspace=0.50, wspace=0.38)

# 3a – Revenue share donut
ax = axes3[0,0]
seg_rev = rfm.groupby('Segment')['Monetary'].sum().reindex(seg_order).dropna().sort_values()
wedges, _, ats = ax.pie(seg_rev.values, colors=[SEG_COLORS[s] for s in seg_rev.index],
                         autopct=lambda p: f'{p:.0f}%' if p>4 else '',
                         startangle=140, pctdistance=0.72,
                         wedgeprops=dict(width=0.55, edgecolor='white', linewidth=1.5))
for at in ats: at.set_fontsize(8); at.set_color('white'); at.set_fontweight('bold')
ax.set_title('Revenue Share by Segment', fontweight='bold')
patches = [mpatches.Patch(color=SEG_COLORS[s], label=f"{s}  £{v:,.0f}") for s,v in zip(seg_rev.index,seg_rev.values)]
ax.legend(handles=patches, loc='center', bbox_to_anchor=(0.5,-0.15), ncol=1, fontsize=7, frameon=False)

# 3b – Monthly trend
ax  = axes3[0,1]
axb = ax.twinx()
df['Month'] = df['InvoiceDate'].dt.to_period('M')
monthly = df.groupby('Month').agg(Orders=('Revenue','count'), Revenue=('Revenue','sum')).reset_index()
monthly['Month_dt'] = monthly['Month'].dt.to_timestamp()
ax.bar(monthly['Month_dt'], monthly['Revenue'], color=TEAL, alpha=0.6, width=20, label='Revenue')
axb.plot(monthly['Month_dt'], monthly['Orders'], color=HIGHLIGHT, lw=2.5, marker='o', ms=5, label='Orders')
ax.set_ylabel('Revenue (£)', color=TEAL); axb.set_ylabel('# Orders', color=HIGHLIGHT)
ax.set_title('Monthly Revenue & Orders', fontweight='bold')
ax.tick_params(axis='x', rotation=35)
l1,n1 = ax.get_legend_handles_labels(); l2,n2 = axb.get_legend_handles_labels()
ax.legend(l1+l2, n1+n2, fontsize=9, loc='upper left')
axb.spines['right'].set_visible(True)

# 3c – RFM heatmap
ax = axes3[0,2]
heat = rfm.groupby(['R_Score','F_Score'])['Monetary'].mean().unstack(fill_value=0)
cmap3 = LinearSegmentedColormap.from_list('rfm',['#f8f9fa','#48cae4','#0f3460'])
sns.heatmap(heat, ax=ax, cmap=cmap3, annot=True, fmt='.0f',
            linewidths=0.5, linecolor='white', cbar_kws={'label':'Avg £'})
ax.set_xlabel('Frequency Score'); ax.set_ylabel('Recency Score')
ax.set_title('Avg Revenue: Recency × Frequency', fontweight='bold')

# 3d – Campaign recommendations (text panel)
ax = axes3[1,0]
ax.set_facecolor('#f0f4f8'); ax.axis('off')
actions = [
    ('Champions',         '#2d6a4f', 'Reward & upsell. Referral incentives.'),
    ('Loyal Customers',   '#40916c', 'Cross-sell. Exclusive early access.'),
    ('At Risk',           '#e94560', '⚠ Win-back + urgency offer NOW.'),
    ('Hibernating',       '#f4845f', '⚠ "We miss you" reactivation email.'),
    ('New Customers',     '#0f3460', 'Onboarding journey + brand story.'),
    ('Potential Loyalist','#48cae4', 'Membership offer. Bundle deals.'),
    ('Lost',              '#adb5bd', 'Low-cost only. Shift budget elsewhere.'),
]
ax.set_title('Campaign Recommendations', fontweight='bold', loc='left', pad=10)
for i,(seg,col,action) in enumerate(actions):
    y = 0.90 - i*0.125
    ax.add_patch(mpatches.FancyBboxPatch((0.01,y-0.05),0.22,0.09,
        boxstyle='round,pad=0.01', facecolor=col, alpha=0.85, transform=ax.transAxes))
    ax.text(0.12, y, seg, transform=ax.transAxes,
            va='center', ha='center', fontsize=8, color='white', fontweight='bold')
    ax.text(0.26, y, action, transform=ax.transAxes,
            va='center', ha='left', fontsize=8, color=PRIMARY)

# 3e – Segment avg metrics table
ax = axes3[1,1]
ax.axis('off')
summary = (rfm.groupby('Segment')
              .agg(Customers=('CustomerID','count'),
                   Avg_Recency=('Recency','mean'),
                   Avg_Frequency=('Frequency','mean'),
                   Avg_Revenue=('Monetary','mean'),
                   Churn_Prob=('Churn_Prob','mean'))
              .reindex(seg_order).dropna().reset_index())
summary['Avg_Recency']   = summary['Avg_Recency'].map('{:.0f}d'.format)
summary['Avg_Frequency'] = summary['Avg_Frequency'].map('{:.1f}'.format)
summary['Avg_Revenue']   = summary['Avg_Revenue'].map('£{:.0f}'.format)
summary['Churn_Prob']    = summary['Churn_Prob'].map('{:.0%}'.format)
summary.columns = ['Segment','N','Recency','Freq','Revenue','Churn%']
tbl2 = ax.table(cellText=summary.values, colLabels=summary.columns,
                cellLoc='center', loc='center')
tbl2.auto_set_font_size(False); tbl2.set_fontsize(8); tbl2.scale(1,1.35)
for (r,c), cell in tbl2.get_celld().items():
    if r==0: cell.set_facecolor(PRIMARY); cell.set_text_props(color='white', weight='bold')
    elif r%2==0: cell.set_facecolor('#f0f4f8')
    cell.set_edgecolor('white')
ax.set_title('Segment Summary Metrics', fontweight='bold', pad=12)

# 3f – KPI cards
ax = axes3[1,2]
ax.axis('off'); ax.set_facecolor('#f0f4f8')
champs   = rfm[rfm['Segment']=='Champions']
at_risk  = rfm[rfm['Segment']=='At Risk']
kpis = [
    ('Total Customers',      f"{len(rfm):,}",                                PRIMARY),
    ('Total Revenue',        f"£{rfm['Monetary'].sum():,.0f}",                ACCENT2),
    ('Avg Order Value',      f"£{rfm['Monetary'].mean():,.0f}",               TEAL),
    ('Churn Rate',           f"{rfm['Churned'].mean():.0%}",                  HIGHLIGHT),
    ('Champions',            f"{len(champs)} ({len(champs)/len(rfm):.0%})",   GREEN),
    ('At-Risk Customers',    f"{len(at_risk)} ({len(at_risk)/len(rfm):.0%})", ORANGE),
    ('Model AUC',            f"{auc:.3f}",                                    GOLD),
]
ax.set_title('Key Metrics', fontweight='bold', loc='left', pad=10)
for i,(label,value,color) in enumerate(kpis):
    y = 0.88 - i*0.125
    ax.add_patch(mpatches.FancyBboxPatch((0.02,y-0.048),0.95,0.09,
        boxstyle='round,pad=0.01', facecolor=color, alpha=0.12, transform=ax.transAxes))
    ax.text(0.06, y, label, transform=ax.transAxes, va='center', fontsize=8.5, color=GREY)
    ax.text(0.94, y, value, transform=ax.transAxes, va='center', ha='right',
            fontsize=9.5, fontweight='bold', color=color)
    
fig1.savefig('rfm_marketing_and_retention_insights.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
rfm.to_csv('rfm_segments.csv', index=False)